# Environment Setup

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["TMPDIR"] = "/data/khawlachrifialaoui/tmp"
os.makedirs(os.environ["TMPDIR"], exist_ok=True)

# Loading the CelebA Dataset

In [ ]:
import kagglehub
import glob
celeba_path = kagglehub.dataset_download("jessicali9530/celeba-dataset")
print("CelebA downloaded at:", celeba_path)

DATA_DIR = celeba_path + "/img_align_celeba"


print("Found", len(DATA_DIR), "images")

/home/khawlachrifialaoui/miniconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CelebA downloaded at: /home/khawlachrifialaoui/.cache/kagglehub/datasets/jessicali9530/celeba-dataset/versions/2
Found 107 images


In [ ]:
OUTPUT_DIR = "/data/khawlachrifialaoui/diffusion_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Libraries and Device Setup

In [ ]:
import math
import glob
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# Data Preprocessing and DataLoader

In [ ]:
IMG_SIZE = 64
BATCH_SIZE = 64

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),  # [-1, 1]
])
class ImageFolderDataset(Dataset):
    def __init__(self, root, transform):
        self.paths = []
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            self.paths.extend(glob.glob(str(Path(root) / "**" / ext), recursive=True))
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img)

dataset = ImageFolderDataset(DATA_DIR, transform)
print(f"Found {len(dataset)} images in {DATA_DIR}")
if len(dataset) == 0:
    raise RuntimeError("No images found. Check DATA_DIR.")

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

Found 202599 images in /home/khawlachrifialaoui/.cache/kagglehub/datasets/jessicali9530/celeba-dataset/versions/2/img_align_celeba


# Diffusion Schedule Initialization

In [ ]:
T = 1000  # number of diffusion steps

betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = torch.cat([torch.tensor([1.0], device=device), alphas_cumprod[:-1]], dim=0)

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)


# Sinusoidal Time Embedding

In [ ]:
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        # t: (B,)
        half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb

# Residual Block and Downsampling Layer

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_emb_dim, groups=8):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_channels)
        self.block1 = nn.Sequential(
            nn.GroupNorm(groups, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, 3, padding=1)
        )
        self.block2 = nn.Sequential(
            nn.GroupNorm(groups, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1)
        )
        if in_channels != out_channels:
            self.res_conv = nn.Conv2d(in_channels, out_channels, 1)
        else:
            self.res_conv = nn.Identity()

    def forward(self, x, t_emb):
        h = self.block1(x)
        t_added = self.time_mlp(t_emb)[:, :, None, None]
        h = h + t_added
        h = self.block2(h)
        return h + self.res_conv(x)

class Downsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 4, 2, 1)

    def forward(self, x):
        return self.conv(x)


# Upsampling Layer

In [ ]:
class Upsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.ConvTranspose2d(channels, channels, 4, 2, 1)

    def forward(self, x):
        return self.conv(x)

# UNet Architecture and Model Initialization

In [ ]:
class UNet(nn.Module):
    def __init__(self, img_channels=3, base_channels=64, time_dim=256):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(time_dim),
            nn.Linear(time_dim, time_dim * 4),
            nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim)
        )

        # Encoder
        self.conv_in = nn.Conv2d(img_channels, base_channels, 3, padding=1)
        self.down1 = ResidualBlock(base_channels, base_channels, time_dim)
        self.down2 = ResidualBlock(base_channels, base_channels*2, time_dim)
        self.downsample1 = Downsample(base_channels*2)
        self.down3 = ResidualBlock(base_channels*2, base_channels*4, time_dim)
        self.downsample2 = Downsample(base_channels*4)

        # Bottleneck
        self.bot1 = ResidualBlock(base_channels*4, base_channels*4, time_dim)
        self.bot2 = ResidualBlock(base_channels*4, base_channels*4, time_dim)

        # Decoder
        self.up1 = Upsample(base_channels*4)
        self.up_block1 = ResidualBlock(base_channels*4 + base_channels*4, base_channels*2, time_dim)
        self.up2 = Upsample(base_channels*2)
        self.up_block2 = ResidualBlock(base_channels*2 + base_channels*2, base_channels, time_dim)
        self.up_block3 = ResidualBlock(base_channels + base_channels, base_channels, time_dim)

        self.conv_out = nn.Sequential(
            nn.GroupNorm(8, base_channels),
            nn.SiLU(),
            nn.Conv2d(base_channels, img_channels, 3, padding=1)
        )

    def forward(self, x, t):
        t_emb = self.time_mlp(t)

        x0 = self.conv_in(x)
        x1 = self.down1(x0, t_emb)                 # 64x64
        x2 = self.down2(x1, t_emb)                 # 64x64
        x3 = self.downsample1(x2)                  # 32x32
        x4 = self.down3(x3, t_emb)                 # 32x32
        x5 = self.downsample2(x4)                  # 16x16

        mid = self.bot1(x5, t_emb)
        mid = self.bot2(mid, t_emb)

        u1 = self.up1(mid)                         # 32x32
        u1 = torch.cat([u1, x4], dim=1)
        u1 = self.up_block1(u1, t_emb)

        u2 = self.up2(u1)                          # 64x64
        u2 = torch.cat([u2, x2], dim=1)
        u2 = self.up_block2(u2, t_emb)

        u3 = torch.cat([u2, x1], dim=1)
        u3 = self.up_block3(u3, t_emb)

        return self.conv_out(u3)

model = UNet().to(device)
print("Model params:", sum(p.numel() for p in model.parameters())/1e6, "M")

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)


Model params: 8.174979 M


# Forward and Reverse Diffusion Process

In [ ]:
def q_sample(x_start, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x_start)
    sqrt_alphas_cum = sqrt_alphas_cumprod[t][:, None, None, None]
    sqrt_one_minus = sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
    return sqrt_alphas_cum * x_start + sqrt_one_minus * noise

def p_sample(model, x, t):
    betas_t = betas[t][:, None, None, None]
    sqrt_one_minus = sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
    sqrt_recip_alpha_t = sqrt_recip_alphas[t][:, None, None, None]

    # Predict noise
    eps_theta = model(x, t.float())

    # Eq. (11) in DDPM: mean of q(x_{t-1} | x_t, x_0)
    model_mean = sqrt_recip_alpha_t * (x - betas_t / sqrt_one_minus * eps_theta)

    if (t == 0).all():
        return model_mean
    else:
        noise = torch.randn_like(x)
        posterior_var = posterior_variance[t][:, None, None, None]
        return model_mean + torch.sqrt(posterior_var) * noise

@torch.no_grad()
def p_sample_loop(model, shape, num_steps=T, save_path=None):
    model.eval()
    x = torch.randn(shape, device=device)
    for i in reversed(range(num_steps)):
        t = torch.full((shape[0],), i, device=device, dtype=torch.long)
        x = p_sample(model, x, t)
    x = (x.clamp(-1, 1) + 1) / 2.0  # back to [0,1]
    grid = utils.make_grid(x.cpu(), nrow=4)
    if save_path is not None:
        plt.figure(figsize=(6,6))
        plt.axis("off")
        plt.imshow(grid.permute(1,2,0))
        plt.savefig(save_path, bbox_inches="tight")
        plt.close()
    return grid


# Model Checkpoint Loading

In [ ]:
import os
import torch

# same model/optimizer definition as before
model = UNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

# --- load latest checkpoint ---
ckpts = [f for f in os.listdir(OUTPUT_DIR) if f.startswith("model_epoch_") and f.endswith(".pt")]
if len(ckpts) == 0:
    start_epoch = 1
    print("No checkpoint found, starting from scratch.")
else:
    # pick the checkpoint with the highest epoch number
    ckpts_sorted = sorted(
        ckpts,
        key=lambda x: int(x.split("_")[-1].split(".")[0])
    )
    last_ckpt = ckpts_sorted[-1]
    ckpt_path = os.path.join(OUTPUT_DIR, last_ckpt)
    print("Loading checkpoint:", ckpt_path)

    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch = ckpt["epoch"] + 1
    print("Resuming from epoch", start_epoch)


Loading checkpoint: /data/khawlachrifialaoui/diffusion_outputs/model_epoch_50.pt
Resuming from epoch 51


# DDPM Training Loop

In [ ]:
EPOCHS = 80
LOG_INTERVAL = 100

global_step = 0

for epoch in range(start_epoch, EPOCHS+1):
    model.train()
    epoch_loss = 0.0

    for batch_idx, x in enumerate(loader):
        x = x.to(device)

        b = x.size(0)
        t = torch.randint(0, T, (b,), device=device).long()
        noise = torch.randn_like(x)

        x_noisy = q_sample(x, t, noise)
        pred_noise = model(x_noisy, t.float())

        loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        global_step += 1

        if batch_idx % LOG_INTERVAL == 0:
            print(f"Epoch {epoch} [{batch_idx}/{len(loader)}] - loss: {loss.item():.4f}")

    avg_loss = epoch_loss / len(loader)
    print(f"Epoch {epoch}/{EPOCHS} - avg loss: {avg_loss:.4f}")
    # Sample some images every few epochs
    if epoch % 5 == 0 or epoch == 1:
        sample_path = os.path.join(OUTPUT_DIR, f"diffusion_epoch_{epoch}.png")
        print(f"Sampling and saving to {sample_path} ...")
        _ = p_sample_loop(model, (16, 3, IMG_SIZE, IMG_SIZE), save_path=sample_path)

    # Optional: save checkpoint
    ckpt_path = os.path.join(OUTPUT_DIR, f"model_epoch_{epoch}.pt")
    torch.save({"model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch}, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}") # 6H:30 for 24 EPOCHS


Epoch 51 [0/3166] - loss: 0.0097
Epoch 51 [100/3166] - loss: 0.0137
Epoch 51 [200/3166] - loss: 0.0210
Epoch 51 [300/3166] - loss: 0.0209
Epoch 51 [400/3166] - loss: 0.0254
Epoch 51 [500/3166] - loss: 0.0127
Epoch 51 [600/3166] - loss: 0.0176
Epoch 51 [700/3166] - loss: 0.0236
Epoch 51 [800/3166] - loss: 0.0186
Epoch 51 [900/3166] - loss: 0.0242
Epoch 51 [1000/3166] - loss: 0.0196
Epoch 51 [1100/3166] - loss: 0.0221
Epoch 51 [1200/3166] - loss: 0.0144
Epoch 51 [1300/3166] - loss: 0.0130
Epoch 51 [1400/3166] - loss: 0.0127
Epoch 51 [1500/3166] - loss: 0.0142
Epoch 51 [1600/3166] - loss: 0.0148
Epoch 51 [1700/3166] - loss: 0.0199
Epoch 51 [1800/3166] - loss: 0.0152
Epoch 51 [1900/3166] - loss: 0.0144
Epoch 51 [2000/3166] - loss: 0.0184
Epoch 51 [2100/3166] - loss: 0.0175
Epoch 51 [2200/3166] - loss: 0.0126
Epoch 51 [2300/3166] - loss: 0.0182
Epoch 51 [2400/3166] - loss: 0.0092
Epoch 51 [2500/3166] - loss: 0.0230
Epoch 51 [2600/3166] - loss: 0.0141
Epoch 51 [2700/3166] - loss: 0.0134
Epoc

# Evaluation Metrics (FID & Inception Score)

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import inception_v3, Inception_V3_Weights
from torchvision.models.inception import InceptionOutputs
from torchvision.transforms import Resize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Main device (diffusion model):", device)

inception_device = torch.device("cpu")
print("Inception device:", inception_device)

model = model.to(device)
model.eval()

torch.cuda.empty_cache()

def denorm_01(x):
    """[-1,1] -> [0,1]"""
    return (x + 1) / 2

def compute_mu_sigma(feats: torch.Tensor):
    mu = feats.mean(dim=0)
    xm = feats - mu
    sigma = xm.T @ xm / (feats.size(0) - 1)
    return mu, sigma

def matrix_sqrt(m: torch.Tensor):
    m = (m + m.T) / 2
    eigvals, eigvecs = torch.linalg.eigh(m)
    eigvals = torch.clamp(eigvals, min=0)
    sqrt_eigvals = torch.sqrt(eigvals)
    return eigvecs @ torch.diag(sqrt_eigvals) @ eigvecs.T

def fid_from_mus_sigmas(mu1, sigma1, mu2, sigma2):
    diff = mu1 - mu2
    covmean = matrix_sqrt(sigma1 @ sigma2)
    fid = diff @ diff + torch.trace(sigma1 + sigma2 - 2 * covmean)
    return fid.item()

def inception_score_from_probs(probs: torch.Tensor, splits: int = 5):
    """probs: N x 1000 tensor of softmax outputs."""
    N = probs.size(0)
    split_size = N // splits
    scores = []

    for i in range(splits):
        part = probs[i * split_size : (i + 1) * split_size]
        py = part.mean(dim=0)  # p(y)
        kl = part * (torch.log(part + 1e-8) - torch.log(py[None, :] + 1e-8))
        kl = kl.sum(dim=1).mean()
        scores.append(torch.exp(kl))

    scores = torch.stack(scores)
    return scores.mean().item(), scores.std().item()

os.environ["TORCH_HOME"] = "/data/khawlachrifialaoui/torch_cache"
os.makedirs(os.environ["TORCH_HOME"], exist_ok=True)

weights = Inception_V3_Weights.IMAGENET1K_V1

inception_logits = inception_v3(weights=weights).to(inception_device)
inception_logits.eval()

inception_features = inception_v3(weights=weights).to(inception_device)
inception_features.fc = nn.Identity()
inception_features.eval()

resize_299 = Resize((299, 299))

@torch.no_grad()
def get_features(dataloader_or_tensor, model, max_images=200, is_tensor=False):
    """
    Returns N x D tensor of pooled features from Inception.
    Assumes images from loader are in [-1,1].
    """
    feats = []

    if is_tensor:
        x = dataloader_or_tensor
        x = denorm_01(x).clamp(0, 1)
        x = resize_299(x)
        x = x.to(inception_device)
        out = model(x)
        if isinstance(out, InceptionOutputs) or isinstance(out, tuple):
            out = out.logits if hasattr(out, "logits") else out[0]
        feats.append(out.cpu())
    else:
        seen = 0
        for x in dataloader_or_tensor:
            x = x.to(inception_device)
            x = denorm_01(x).clamp(0, 1)
            x = resize_299(x)
            out = model(x)
            if isinstance(out, InceptionOutputs) or isinstance(out, tuple):
                out = out.logits if hasattr(out, "logits") else out[0]
            feats.append(out.cpu())
            seen += x.size(0)
            if seen >= max_images:
                break

    feats = torch.cat(feats, dim=0)
    if feats.size(0) > max_images:
        feats = feats[:max_images]
    return feats

@torch.no_grad()
def sample_diffusion_batch(model, bs):
    """
    Generate `bs` fake images using the diffusion process.
    Uses tiny batch size to avoid CUDA OOM.
    Returns tensor [bs,3,IMG_SIZE,IMG_SIZE] in [0,1].
    """
    model.eval()
    x = torch.randn((bs, 3, IMG_SIZE, IMG_SIZE), device=device)

    for i in reversed(range(T)):
        t = torch.full((bs,), i, device=device, dtype=torch.long)
        x = p_sample(model, x, t)
        torch.cuda.empty_cache()

    x = (x.clamp(-1, 1) + 1) / 2.0  # [-1,1] → [0,1]
    return x

NUM_SAMPLES = 128
BATCH_FAKE  = 1

print("\n=== Computing FID & IS for diffusion model (light mode) ===")

print("Collecting real features for FID (diffusion)...")
real_feats = get_features(loader, inception_features, max_images=NUM_SAMPLES, is_tensor=False)
mu_real, sigma_real = compute_mu_sigma(real_feats)
print(f"Collected {real_feats.size(0)} real features.")

print("Sampling diffusion images and collecting features/probs...")
fake_feats_list = []
probs_list      = []
fake_seen       = 0

while fake_seen < NUM_SAMPLES:
    bs = min(BATCH_FAKE, NUM_SAMPLES - fake_seen)

    fake_batch = sample_diffusion_batch(model, bs)

    x_in = resize_299(fake_batch.clamp(0, 1)).to(inception_device)

    # Features for FID
    out_f = inception_features(x_in)
    if isinstance(out_f, InceptionOutputs) or isinstance(out_f, tuple):
        out_f = out_f.logits if hasattr(out_f, "logits") else out_f[0]
    fake_feats_list.append(out_f.cpu())

    # Logits for IS
    out_l = inception_logits(x_in)
    if isinstance(out_l, InceptionOutputs) or isinstance(out_l, tuple):
        out_l = out_l.logits if hasattr(out_l, "logits") else out_l[0]
    p_yx = F.softmax(out_l, dim=1)
    probs_list.append(p_yx.cpu())

    fake_seen += bs
    print(f"  Generated {fake_seen}/{NUM_SAMPLES} fake images...", end="\r")

    del fake_batch, x_in, out_f, out_l, p_yx
    torch.cuda.empty_cache()

fake_feats = torch.cat(fake_feats_list, dim=0)[:NUM_SAMPLES]
mu_fake, sigma_fake = compute_mu_sigma(fake_feats)

probs_fake = torch.cat(probs_list, dim=0)[:NUM_SAMPLES]

fid_value = fid_from_mus_sigmas(mu_real, sigma_real, mu_fake, sigma_fake)
is_mean, is_std = inception_score_from_probs(probs_fake, splits=4)

print("\n===== FINAL METRICS (DIFFUSION, LIGHT) =====")
print(f"FID: {fid_value:.2f}")
print(f"Inception Score: {is_mean:.2f} ± {is_std:.2f}")
print("============================================")


Main device (diffusion model): cuda
Inception device: cpu

=== Computing FID & IS for diffusion model (light mode) ===
Collected 128 real features.
Sampling diffusion images and collecting features/probs...
  Generated 128/128 fake images...
===== FINAL METRICS (DIFFUSION, LIGHT) =====
FID: 56.79
Inception Score: 1.90 ± 0.14
